In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [4]:
louisiana

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns
0,0,louisiana,0,0,367983.8318,68239.85746,80.234375,79198.54166,6714.566582,1154589.900,...,9.980000,0.322352,2.821843,4.032755,0.0,39.360000,15.368289,106.770000,4.290000,32.920000
1,0,louisiana,1,0,362385.6368,67201.71394,79.013757,77993.68198,6612.416842,1137024.945,...,9.968923,0.319496,2.800291,4.092947,0.0,40.166298,14.997723,108.605133,3.731796,31.374749
2,0,louisiana,2,0,361096.1267,66962.58391,78.732596,77716.14990,6588.887271,1132978.965,...,9.489254,0.320059,2.788854,4.099808,0.0,38.639666,13.763035,80.719306,3.559992,31.843131
3,0,louisiana,3,0,359935.7476,66747.40026,78.479589,77466.40976,6567.713942,1129338.147,...,10.279229,0.321689,2.777719,4.162110,0.0,41.465412,14.851001,146.300301,3.841408,33.650089
4,0,louisiana,4,0,358773.5212,66531.87404,78.226180,77216.27204,6546.506906,1125691.534,...,9.983308,0.323218,2.772568,4.173150,0.0,37.807488,13.488061,146.740554,3.879013,34.227053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2407,127127,louisiana,31,0,311979.6878,53280.76885,83.099337,74341.21831,6871.576034,1116082.754,...,2.290044,0.099194,1.610628,2.566150,0.0,9.504129,1.357871,6.354095,0.494434,17.846785
2408,127127,louisiana,32,0,310670.5663,53049.87938,83.075690,74040.77745,6869.511396,1114713.368,...,2.285685,0.087899,1.527726,2.529939,0.0,9.783970,1.280190,4.725632,0.364936,17.059667
2409,127127,louisiana,33,0,309356.4650,52821.26235,83.037733,73734.23638,6866.309713,1113193.917,...,2.281655,0.076939,1.445208,2.497102,0.0,9.535228,1.030977,3.223424,0.239272,16.254621
2410,127127,louisiana,34,0,308044.9743,52594.98521,82.990728,73425.33726,6862.387927,1111581.984,...,2.277804,0.066352,1.363483,2.466951,0.0,9.696216,0.918047,1.841394,0.117569,15.427843


In [ ]:
import pandas as pd

# 1) Filter base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()

# 2) List your transport fuels
relevant_fuels = [
    'diesel',
    'electricity',
    'gasoline',
    'hydrocarbon_gas_liquids',
    'hydrogen',
    'natural_gas'
]

# 3) Quick check of available demand & efficiency columns
print("=== available transport-demand columns ===")
for c in base_case.columns:
    if 'trns_fuel_' in c:
        print(" ", c)
print("\n=== available transport-efficiency columns ===")
for c in base_case.columns:
    if 'fuelefficiency_trns_road_light' in c:
        print(" ", c)
print("\n")

# 4) Prepare accumulators
total_saved_volume = pd.Series(0.0, index=base_case.index)
demand_trns = pd.DataFrame({'time_period': base_case['time_period']},
                           index=base_case.index)

# 5) Loop & pattern-match per fuel
for fuel in relevant_fuels:
    # 5a) demand cols specific to this fuel
    dem_cols = [
        c for c in base_case.columns
        if f'trns_fuel_{fuel}' in c
    ]
    # 5b) efficiency cols for this fuel
    eff_cols = [
        c for c in base_case.columns
        if f'fuelefficiency_trns_road_light_{fuel}' in c
    ]

    print(f"Fuel={fuel!r}: dem_cols={dem_cols}, eff_cols={eff_cols}")
    if not dem_cols or not eff_cols:
        print(f"  → skipping {fuel!r} (no matching columns)\n")
        continue

    fuel_demand     = base_case[dem_cols[0]]
    fuel_efficiency = base_case[eff_cols[0]]

    demand_trns[fuel] = fuel_demand

    vol_now  = fuel_demand / fuel_efficiency
    vol_base = fuel_demand / fuel_efficiency.iloc[0]
    total_saved_volume += (vol_base - vol_now)

# 6) Build output (with identifiers)
output_trns = pd.DataFrame({
    'primary_id': base_case['primary_id'],
    'region':     base_case['region'],
    'time_period': base_case['time_period'],
    'transportation_volume_saved_in_PJ': total_saved_volume
}, index=base_case.index)

# 7) Apply $880 000 per PJ
capex_multiplier_trns = 880_000
output_trns['transportation_efficiency_capex'] = (
    output_trns['transportation_volume_saved_in_PJ'] * capex_multiplier_trns
)
output_trns['transportation_efficiency_opex'] = 0

# 8) (Optional) save
#output_trns.to_csv('transportation_energy_cost.csv', index=False)


=== available transport-demand columns ===
  energy_demand_enfu_subsector_total_pj_trns_fuel_ammonia
  energy_demand_enfu_subsector_total_pj_trns_fuel_biofuels
  energy_demand_enfu_subsector_total_pj_trns_fuel_biogas
  energy_demand_enfu_subsector_total_pj_trns_fuel_biomass
  energy_demand_enfu_subsector_total_pj_trns_fuel_coal
  energy_demand_enfu_subsector_total_pj_trns_fuel_coke
  energy_demand_enfu_subsector_total_pj_trns_fuel_crude
  energy_demand_enfu_subsector_total_pj_trns_fuel_diesel
  energy_demand_enfu_subsector_total_pj_trns_fuel_electricity
  energy_demand_enfu_subsector_total_pj_trns_fuel_furnace_gas
  energy_demand_enfu_subsector_total_pj_trns_fuel_gasoline
  energy_demand_enfu_subsector_total_pj_trns_fuel_geothermal
  energy_demand_enfu_subsector_total_pj_trns_fuel_hydrocarbon_gas_liquids
  energy_demand_enfu_subsector_total_pj_trns_fuel_hydrogen
  energy_demand_enfu_subsector_total_pj_trns_fuel_kerosene
  energy_demand_enfu_subsector_total_pj_trns_fuel_natural_gas
  en

In [9]:
output_trns

,primary_id,region,time_period,transportation_volume_saved_in_PJ,transportation_efficiency_capex,transportation_efficiency_opex
0,0,louisiana,0,0.000000,0.000000e+00,0
1,0,louisiana,1,0.014345,1.262368e+04,0
2,0,louisiana,2,0.029470,2.593372e+04,0
3,0,louisiana,3,0.050429,4.437719e+04,0
4,0,louisiana,4,0.077275,6.800232e+04,0
5,0,louisiana,5,0.130341,1.146998e+05,0
6,0,louisiana,6,0.162717,1.431912e+05,0
7,0,louisiana,7,0.589068,5.183801e+05,0
8,0,louisiana,8,1.025977,9.028602e+05,0
9,0,louisiana,9,1.475988,1.298870e+06,0
